# Simulated patient heart model generation

This notebook generates *simulated* per-patient heart meshes by scaling each segmented component of the healthy heart **GLB** according to clinical conditions.

Inputs:
- `data_processing/hvsmr_clinical.csv`
- `models/condition_effects.json`
- `heart_models/cardiac_conduction_system.glb` (segmented by mesh names)

Outputs:
- `heart_models/simulated_patient_models/simulated_patient{id}.obj`


In [2]:
# Imports + paths
import json
from pathlib import Path

import numpy as np
import pandas as pd

# Find project root by walking up from CWD
PROJECT_MARKERS = [
    Path('models') / 'condition_effects.json',
    Path('heart_models') / 'cardiac_conduction_system.glb',
    Path('data_processing') / 'hvsmr_clinical.csv',
]

def find_project_root(start: Path, markers=PROJECT_MARKERS, max_up: int = 8) -> Path:
    start = start.resolve()
    cur = start
    for _ in range(max_up + 1):
        if any((cur / m).exists() for m in markers):
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    raise FileNotFoundError(
        f"Could not infer PROJECT_ROOT from {start}. Looked for markers: {markers}"
    )

PROJECT_ROOT = find_project_root(PROJECT_ROOT)
DATA_DIR = PROJECT_ROOT / 'data-processing'

clinical_path = DATA_DIR / 'hvsmr_clinical.csv'
if not clinical_path.exists():
    raise FileNotFoundError(f'Missing: {clinical_path}')

hvsmr_clinical = pd.read_csv(clinical_path)

condition_json_path = PROJECT_ROOT / 'models' / 'condition_effects.json'
if not condition_json_path.exists():
    raise FileNotFoundError(f'Missing: {condition_json_path}')

with open(condition_json_path, 'r', encoding='utf-8') as f:
    condition_effects = json.load(f)

cond_mult = condition_effects['condition_multipliers']

heart_models_dir = PROJECT_ROOT / 'heart_models'
healthy_segmented_glb = heart_models_dir / 'cardiac_conduction_system.glb'
if not healthy_segmented_glb.exists():
    raise FileNotFoundError(f'Missing: {healthy_segmented_glb}')

out_dir = heart_models_dir / 'simulated_patient_models'
out_dir.mkdir(parents=True, exist_ok=True)

print('CWD:', PROJECT_ROOT)
print('Project root:', PROJECT_ROOT)
print('Loaded clinical rows:', hvsmr_clinical.shape)
print('Healthy segmented GLB:', healthy_segmented_glb)
print('Output dir:', out_dir)


CWD: /Users/vehnilrangaraman/Desktop/COLLEGE/HACKLYTICS/2026/hacklytics2026/notebooks
Project root: /Users/vehnilrangaraman/Desktop/COLLEGE/HACKLYTICS/2026/hacklytics2026
Loaded clinical rows: (59, 36)
Healthy segmented GLB: /Users/vehnilrangaraman/Desktop/COLLEGE/HACKLYTICS/2026/hacklytics2026/heart_models/cardiac_conduction_system.glb
Output dir: /Users/vehnilrangaraman/Desktop/COLLEGE/HACKLYTICS/2026/hacklytics2026/heart_models/simulated_patient_models


In [3]:
# Frontend-equivalent mapping + helpers

# From heart-scene.js
PART_NAME_TO_VOL_KEY = {
    'LV': 'Label_1_vol_ml',
    'RV': 'Label_2_vol_ml',
    'LA': 'Label_3_vol_ml',
    'RA': 'Label_4_vol_ml',
    'AO': 'Label_5_vol_ml',
    'PA': 'Label_6_vol_ml',
    'PV': 'Label_7_vol_ml',
    'SVC': 'Label_8_vol_ml',
}

# Map GLB mesh names -> these part codes (adjust as needed)
GLB_MESH_TO_PART = {
    'Aorta_Aorta_0': 'AO',
    'Pulmonary_trunk_Pulmonary_trunk_0': 'PA',
    'Arteries1_Arteries1_0': 'PA',
    'Arteries3_Arteries3_0': 'PA',
    'Arteries5_Arteries5_0': 'PA',
    'Av_valves_Av_valves_0': None,
    'Valves5_Valves5_0': None,
    'Valves2_Valves2_0': None,
    'Ligament_Ligament_0': None,
    'Septum_Septum_0': None,
    'Heart_basis1_Heart_basis1_0': None,
    'Conduction__Conduction__0': None,
}

CORE_COLS = {'Pat', 'Age', 'Category'}
condition_cols = [c for c in hvsmr_clinical.columns if c not in CORE_COLS]


def patient_conditions(pat_id: int):
    row = hvsmr_clinical.loc[hvsmr_clinical['Pat'].astype(int) == int(pat_id)]
    if row.empty:
        raise KeyError(f'Patient {pat_id} not found in hvsmr_clinical')
    row = row.iloc[0]
    conds = []
    for c in condition_cols:
        v = row[c]
        if isinstance(v, str) and v.strip().upper() == 'X':
            conds.append(c)
    return conds


def compute_scales_for_conditions(selected_conditions, default=1.0):
    # Match heart-scene.js computeScalesForConditions:
    # - geometric mean of multipliers per part across conditions
    # - clamp multiplier to [0.2, 5]
    # - convert volume ratio to linear scale via cube root
    if not selected_conditions:
        return {part: float(default) for part in PART_NAME_TO_VOL_KEY.keys()}

    selected_conditions = [c for c in selected_conditions if c != 'Normal']
    scales = {part: float(default) for part in PART_NAME_TO_VOL_KEY.keys()}
    if not selected_conditions:
        return scales

    for part, vol_key in PART_NAME_TO_VOL_KEY.items():
        product = 1.0
        n = 0
        for cond in selected_conditions:
            mult = cond_mult.get(cond, {}).get(vol_key, None)
            if mult is not None and mult > 0:
                product *= float(mult)
                n += 1
        mult = (product ** (1 / n)) if n > 0 else 1.0
        mult = max(0.2, min(5.0, mult))
        scales[part] = float(np.cbrt(mult))
    return scales


In [4]:
# GLB parsing utilities + OBJ writer

# Requires: pygltflib (pip install pygltflib)
from pygltflib import GLTF2
import numpy as np


def _accessor_data(gltf: GLTF2, accessor_idx: int) -> np.ndarray:
    accessor = gltf.accessors[accessor_idx]
    buffer_view = gltf.bufferViews[accessor.bufferView]
    buffer = gltf.buffers[buffer_view.buffer]

    dtype_map = {
        5120: np.int8,
        5121: np.uint8,
        5122: np.int16,
        5123: np.uint16,
        5125: np.uint32,
        5126: np.float32,
    }
    comp_dtype = dtype_map[accessor.componentType]
    type_num = {
        'SCALAR': 1,
        'VEC2': 2,
        'VEC3': 3,
        'VEC4': 4,
        'MAT4': 16,
    }[accessor.type]

    byte_offset = (buffer_view.byteOffset or 0) + (accessor.byteOffset or 0)
    byte_length = accessor.count * type_num * np.dtype(comp_dtype).itemsize

    if buffer.uri:
        # external buffer not expected here
        raise ValueError('External buffer URIs not supported in this loader')

    data = gltf.binary_blob()
    arr = np.frombuffer(data[byte_offset:byte_offset + byte_length], dtype=comp_dtype)
    return arr.reshape((accessor.count, type_num))


def load_glb_meshes(glb_path: Path):
    """Return dict: mesh_name -> (verts, faces) in mesh-local order."""
    gltf = GLTF2().load(str(glb_path))
    meshes = {}
    for mesh in gltf.meshes:
        mesh_name = mesh.name or 'UnnamedMesh'
        # Use first primitive only (common for these assets)
        prim = mesh.primitives[0]
        pos_accessor = prim.attributes.POSITION
        verts = _accessor_data(gltf, pos_accessor).astype(np.float32)

        faces = None
        if prim.indices is not None:
            idx = _accessor_data(gltf, prim.indices).astype(np.int64).ravel()
            faces = idx.reshape((-1, 3))
        else:
            # no indices; assume sequential triangles
            idx = np.arange(len(verts), dtype=np.int64)
            faces = idx.reshape((-1, 3))

        meshes[mesh_name] = (verts, faces)
    return meshes


def scale_vertices(verts: np.ndarray, scale: float):
    if abs(scale - 1.0) < 1e-8:
        return verts
    centroid = verts.mean(axis=0)
    return centroid + (verts - centroid) * scale


def write_obj_grouped(out_path: Path, mesh_dict: dict):
    with open(out_path, 'w', encoding='utf-8') as f:
        f.write('# Simulated patient mesh generated from cardiac_conduction_system.glb\n')
        # global vertex list
        offset = 0
        for name, (verts, _) in mesh_dict.items():
            for v in verts:
                f.write(f'v {v[0]:.6f} {v[1]:.6f} {v[2]:.6f}\n')
        # faces with groups
        for name, (verts, faces) in mesh_dict.items():
            f.write(f'g {name}\n')
            for (a, b, c) in faces:
                f.write(f'f {a+1+offset} {b+1+offset} {c+1+offset}\n')
            offset += verts.shape[0]


In [5]:
# Generate simulated meshes for selected patients

mesh_dict = load_glb_meshes(healthy_segmented_glb)
print('Loaded GLB meshes:', len(mesh_dict))
print('Mesh names:', list(mesh_dict.keys()))

# Generate for all patients by default
SIM_PAT_IDS = sorted(hvsmr_clinical['Pat'].dropna().astype(int).unique().tolist())

rows = []
for pat_id in SIM_PAT_IDS:
    conds = patient_conditions(pat_id)
    part_scales = compute_scales_for_conditions(conds)

    scaled_meshes = {}
    for mesh_name, (verts, faces) in mesh_dict.items():
        part = GLB_MESH_TO_PART.get(mesh_name, None)
        s = float(part_scales.get(part, 1.0)) if part is not None else 1.0
        v_sim = scale_vertices(verts, s)
        scaled_meshes[mesh_name] = (v_sim, faces)

    out_path = out_dir / f'simulated_patient{int(pat_id)}.obj'
    write_obj_grouped(out_path, scaled_meshes)

    row = {
        'pat': int(pat_id),
        'n_conditions': len(conds),
        'conditions': ','.join(conds),
        'out_path': str(out_path),
    }
    for mesh_name in mesh_dict.keys():
        part = GLB_MESH_TO_PART.get(mesh_name, None)
        row[f'scale_{mesh_name}'] = float(part_scales.get(part, 1.0)) if part is not None else 1.0
    rows.append(row)

sim_summary = pd.DataFrame(rows)
print(sim_summary.head(10).to_string(index=False))
print('Wrote', len(sim_summary), 'OBJ(s) to:', out_dir)


Loaded GLB meshes: 12
Mesh names: ['Av_valves_Av_valves_0', 'Ligament_Ligament_0', 'Valves5_Valves5_0', 'Valves2_Valves2_0', 'Pulmonary_trunk_Pulmonary_trunk_0', 'Arteries5_Arteries5_0', 'Arteries1_Arteries1_0', 'Arteries3_Arteries3_0', 'Aorta_Aorta_0', 'Septum_Septum_0', 'Heart_basis1_Heart_basis1_0', 'Conduction__Conduction__0']
 pat  n_conditions                                                     conditions                                                                                                                            out_path  scale_Av_valves_Av_valves_0  scale_Ligament_Ligament_0  scale_Valves5_Valves5_0  scale_Valves2_Valves2_0  scale_Pulmonary_trunk_Pulmonary_trunk_0  scale_Arteries5_Arteries5_0  scale_Arteries1_Arteries1_0  scale_Arteries3_Arteries3_0  scale_Aorta_Aorta_0  scale_Septum_Septum_0  scale_Heart_basis1_Heart_basis1_0  scale_Conduction__Conduction__0
   0             3                                             VSD,DORV,PABanding /Users/vehnilrangaraman/D